# Evaluación del TEST real
Selecciona una corrida final por arquitectura y semilla después de revisar VAL y la referencia pública. La primera ejecución congela la selección completa antes de abrir TEST. Los reintentos reutilizan los resultados terminados.

In [ ]:
from pathlib import Path
from IPython import get_ipython

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()
)


def load_notebook(name):
    path = ROOT / "notebooks" / "shared" / (name + ".ipynb")
    get_ipython().run_line_magic("run", f'-i "{path.as_posix()}"')


load_notebook("workspace")

In [ ]:
load_notebook("experiment")
load_notebook("metrics")
records = []
for path in sorted(RUNS.glob("*/record.json")):
    record = json.loads(path.read_text(encoding="utf-8"))
    if record["stage"] == "baseline" and record["status"] == "PASS":
        records.append(
            (
                path.parent.name,
                record["config"]["model"],
                record["seed"],
                record["dataset_hash"],
            )
        )
display(pd.DataFrame(records, columns=["run_id", "model", "seed", "dataset_hash"]))

In [ ]:
selected_runs = []

In [ ]:
frame, dataset_hash = cap_rows()
suite = RUNS.parent / "final" / dataset_hash
suite.mkdir(parents=True, exist_ok=True)
plan_file = suite / "plan.json"
if plan_file.exists() and not selected_runs:
    selected_runs = list(json.loads(plan_file.read_text())["runs"])
plan = {}
keys = set()
for name in sorted(selected_runs):
    if name not in {row[0] for row in records}:
        raise ValueError("Selecciona únicamente corridas finales completadas.")
    run = RUNS / name
    record = json.loads((run / "record.json").read_text(encoding="utf-8"))
    cfg = record["config"]
    expected = json.loads(
        (ROOT / "configs/models" / (cfg["model"] + ".json")).read_text()
    )
    if (
        cfg != expected
        or record["implementation_hash"] != implementation_hash()
        or record["dataset_hash"] != dataset_hash
        or record["lock_hash"] != digest(ROOT / "uv.lock")
    ):
        raise ValueError(
            "La configuración, el entorno o los datos difieren de los congelados."
        )
    keys.add((cfg["model"], record["seed"]))
    plan[name] = {
        file: digest(run / file)
        for file in ("weights.pt", "record.json", "threshold.json")
    }
required = {
    (name, seed)
    for name in MODELS
    for seed in json.loads((ROOT / "configs/models" / (name + ".json")).read_text())[
        "seeds"
    ]
}
if keys != required or len(plan) != len(required):
    raise ValueError("Se requiere exactamente una corrida por modelo y semilla.")
payload = {"dataset_hash": dataset_hash, "runs": plan}
if plan_file.exists() and json.loads(plan_file.read_text()) != payload:
    raise ValueError("La selección ya fue congelada para este TEST.")
if not plan_file.exists():
    with plan_file.open("x") as stream:
        json.dump(payload, stream, indent=2)
export_study_file(suite, "plan.json")
for name in plan:
    import gc

    run, folder = RUNS / name, suite / name
    if (folder / "metrics.json").exists():
        export_study_file(folder, "metrics.json")
        continue
    cfg = json.loads((run / "record.json").read_text(encoding="utf-8"))["config"]
    model, transform = build_model(cfg)
    model.load_state_dict(
        torch.load(run / "weights.pt", map_location="cpu", weights_only=True)
    )
    torch.cuda.reset_peak_memory_stats()
    threshold = json.loads((run / "threshold.json").read_text())["value"]
    assess(model, transform, frame[frame.split == "test"], cfg, folder, threshold)
    export_study_file(folder, "metrics.json")
    del model
    gc.collect()
    torch.cuda.empty_cache()
print("Evaluación completa:", suite)